In [35]:
from pathlib import Path

# Set this to your acc_cpp directory if the notebook kernel starts elsewhere.
ACC_CPP_DIR = Path("/home/danilo/Desktop/perception_uncertainty/experiments/esmini-perception/simulation/acc_cpp")
SCENARIO_FILE = ACC_CPP_DIR / "../../Data/C_original_1549178.xosc"
BINARY_FILE = ACC_CPP_DIR / "build/runner/acc_sim"
OUTPUT_ROOT = ACC_CPP_DIR / "results"
TIMESTEP_SECONDS = 0.01
DURATION_SECONDS = 10.0
TIME_GAP_SECONDS = 1.5
SET_SPEED_MPS = None  # None uses the scenario's initial ego speed.
STOP_ON_COLLISION = True
DETAILED_LOGS = False
VIEWER = True  # True opens live animation; build first with tools/build_native.py --viewer.
NATIVE_REFERENCE = False  # False: target copies; True: live native targets.

# esmini's ObjectSensor: one measurement per simulation timestep, no uncertainty.
IDEAL_SENSOR_ENABLED = True  # False keeps all-target copies for validation.
SENSOR_X_METRES = 0.0
SENSOR_Y_METRES = 0.0
SENSOR_Z_METRES = 0.5
SENSOR_HEADING_RADIANS = 0.0
SENSOR_NEAR_RANGE_METRES = 0.0
SENSOR_FAR_RANGE_METRES = 100.0
SENSOR_HORIZONTAL_FOV_RADIANS = 1.0471975511965976  # 60 degrees.


In [36]:
import subprocess
import tempfile

binary = BINARY_FILE.resolve()
scenario = SCENARIO_FILE.resolve()
if not binary.is_file() or not scenario.is_file():
    raise FileNotFoundError("Check ACC_CPP_DIR, BINARY_FILE and SCENARIO_FILE in the settings cell.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
output_dir = Path(tempfile.mkdtemp(prefix="acc_single_", dir=OUTPUT_ROOT.resolve()))


# Run the ACC simulation with the specified parameters.
command = [
    str(binary), "--scenario", str(scenario), "--output-dir", str(output_dir),
    "--controller", "acc", "--dropout-p", "0",
    "--dt", str(TIMESTEP_SECONDS), "--duration", str(DURATION_SECONDS),
    "--time-gap", str(TIME_GAP_SECONDS),
    "--stop-on-collision", str(STOP_ON_COLLISION).lower(),
    "--detailed-logs", str(DETAILED_LOGS).lower(),
    "--viewer", str(VIEWER).lower(),
    "--native-reference", str(NATIVE_REFERENCE).lower(),
]
command += [
    "--ideal-sensor", str(IDEAL_SENSOR_ENABLED).lower(),
    "--sensor-x-m", str(SENSOR_X_METRES),
    "--sensor-y-m", str(SENSOR_Y_METRES),
    "--sensor-z-m", str(SENSOR_Z_METRES),
    "--sensor-heading-rad", str(SENSOR_HEADING_RADIANS),
    "--sensor-near-m", str(SENSOR_NEAR_RANGE_METRES),
    "--sensor-far-m", str(SENSOR_FAR_RANGE_METRES),
    "--sensor-horizontal-fov-rad", str(SENSOR_HORIZONTAL_FOV_RADIANS),
]
if SET_SPEED_MPS is not None:
    command += ["--set-speed", str(SET_SPEED_MPS)]
print("Output:", output_dir)
print(f"Using sensor option: ideal={IDEAL_SENSOR_ENABLED}")
subprocess.run(command, cwd=ACC_CPP_DIR, check=True)


Output: /home/danilo/Desktop/perception_uncertainty/experiments/esmini-perception/simulation/acc_cpp/results/acc_single_cp2t9z6c
Using sensor option: ideal=True
esmini GIT REV: v3.8.1-0-19d26b6-dirty
esmini GIT TAG: v3.8.1
esmini GIT BRANCH: tags/v3.8.1^0
esmini BUILD VERSION: N/A - client build
[] [info] Ignoring missing default config: /home/danilo/Desktop/perception_uncertainty/experiments/esmini-perception/simulation/acc_cpp/build/engine-home/bin/../config.yml
Completed: "/home/danilo/Desktop/perception_uncertainty/experiments/esmini-perception/simulation/acc_cpp/results/acc_single_cp2t9z6c"
collision=0, min_gap=17.2805 m, min_TTC=5.1556 s


CompletedProcess(args=['/home/danilo/Desktop/perception_uncertainty/experiments/esmini-perception/simulation/acc_cpp/build/runner/acc_sim', '--scenario', '/home/danilo/Desktop/perception_uncertainty/experiments/esmini-perception/Data/C_original_1549178.xosc', '--output-dir', '/home/danilo/Desktop/perception_uncertainty/experiments/esmini-perception/simulation/acc_cpp/results/acc_single_cp2t9z6c', '--controller', 'acc', '--dropout-p', '0', '--dt', '0.01', '--duration', '10.0', '--time-gap', '1.5', '--stop-on-collision', 'true', '--detailed-logs', 'false', '--viewer', 'true', '--native-reference', 'false', '--ideal-sensor', 'true', '--sensor-x-m', '0.0', '--sensor-y-m', '0.0', '--sensor-z-m', '0.5', '--sensor-heading-rad', '0.0', '--sensor-near-m', '0.0', '--sensor-far-m', '100.0', '--sensor-horizontal-fov-rad', '1.0471975511965976'], returncode=0)

In [ ]:
import csv
import json

with (output_dir / "results.csv").open(newline="") as file:
    results = list(csv.DictReader(file))
with (output_dir / "detections.csv").open(newline="") as file:
    detections = list(csv.DictReader(file))
run_config = json.loads((output_dir / "run_config.json").read_text())
summary = json.loads((output_dir / "summary.json").read_text())
print(f"Loaded {len(results)} results rows")
summary
